[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/4590_Colabs/blob/main/4_CellModels.ipynb)

# Set 4 — Cellular Models & Electrophysiology Basics
**Author: Neural Engineering Laboratory, University of Missouri-Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair**

## Introduction
Welcome to the Cellular Modeling module. This lab is designed to move you from a qualitative understanding of neurons to a **quantitative appreciation** of how they function. We achieve this by exploring the spectrum of neural abstraction: from the computationally efficient **LIF** and **Izhikevich** models used in large-scale networks, to the biophysically detailed **Hodgkin-Huxley (HH)** equations that simulate actual ion channel kinetics.

By the end of this module, you will be able to map biological phenomena (like refractory periods and thresholding) to specific mathematical parameters and observe how measured physical signals can be used to drive simulated neural activity.

> ### NOTE for Learners
> While the notebook begins with simplified models (C1-C2), these abstractions assume a firm grasp of electrophysiology principles. **It is strongly recommended that you complete the [Set 3 Electrophysiology Appendix](https://colab.research.google.com/github/cyneuro/4590_Colabs/blob/main/3_PassiveMembrane.ipynb) (opens in a new notebook) PRIOR to beginning the contents in Set 4.** The Appendix acts as your "Experimental Bench," where you will learn how to measure input resistance ($R_{in}$) and time constants ($\tau$) — concepts essential for interpreting the models in this module.

---
### A Note on This Notebook
**This completed, filled-in notebook is the lab deliverable.** Add your own cells as you go. **C4 — Real-world Data Bridge** below is a good spot for an optional (AI-assisted) extension of your own.

---
### 📋 Before You Begin: How This Module Is Graded
This module's submission and its companion quiz are weighted **equally**. The quiz specifically checks whether the understanding behind your submission is actually yours — pasting an AI's explanation, or a friend's slider values, can get this notebook looking complete, but it won't get you through a quiz question that applies the same concept to a scenario you haven't seen before.

The **design problem** below, and the **question you'll create yourself**, both exist for exactly this reason: real practice at applying and testing these concepts, not just observing them.

---

## Table of Contents
* **[Introduction](#Introduction)** — Module overview and goals.
* **[C0 Starter](#C0-Starter)** — Environment setup (Run this first!).
* **[Set 3 Electrophysiology Appendix](https://colab.research.google.com/github/cyneuro/4590_Colabs/blob/main/3_PassiveMembrane.ipynb)** — **Start Here** to learn the "Experimental Mindset" (opens in a new notebook).
* **[C1 Leaky Integrate-and-Fire (LIF)](#C1-LIF)** — The fundamental RC integrator.
* **[C2 Izhikevich Model](#C2-Izhikevich)** — Adding adaptation and bursting with low cost.
* **[C3 Hodgkin-Huxley via NEURON](#C3-HH)** — Detailed channel kinetics and biophysics.
* **[C4 Real-world Data Bridge](#C4-microbit)** — Mapping sensor data (micro:bit) to current clamp.
* **[Reflection & Discussion](#Reflection)** — Synthesizing biology-to-model mapping.

## C0 Starter — Setup & Readiness Check
Before we begin, we must initialize the programming environment. This cell installs the **NEURON** simulator and ensures the necessary libraries are ready for the Appendix and C3.

### Interactive Module Note
This notebook is designed as an **Interactive Experimental Rig**.
* **Sliders:** Use the sliders on the right of code cells to adjust parameters.
* **Auto-Run:** Most cells are set to `run: "auto"`, meaning the plots will update the moment you move a slider.

In [ ]:
# Install dependencies with quotes around the version constraint
!pip install -q neuron "numpy<2.1" matplotlib --upgrade

import neuron
from neuron import h
import numpy as np
import matplotlib.pyplot as plt

# Ensure standard run functions are available
h.load_file('stdrun.hoc')

print("NEURON is ready and functional.")


## C1 — Leaky Integrate-and-Fire (LIF)
### Interactive Exercise: The Temporal Integrator
The LIF model is a "leaky" bucket for charge. Use the sliders to find the balance between how fast the cell leaks ($tau$) and how much it resists change ($R$).

**Task:** Adjust the **tau** slider to 0.040 (40ms). Does the spike count increase or decrease compared to the baseline? Why?

---
### Exercises: The Temporal Integrator
**Exercise 1: Time Constant Impact**
1. **Predict:** If you double the time constant ($\tau$) from 20ms to 40ms, will the neuron require a *higher* or *lower* frequency of input pulses to reach the threshold?
2. **Verify:** Change `tau = 40e-3` in the code cell above. Observe the "Spikes" count in the output. Did it change as you expected?

**Exercise 2: Resistance vs. Threshold**
1. **Predict:** If you increase the Input Resistance ($R$) from 100 to 150, will the "Spikes" count increase or decrease?
2. **Verify:** Adjust $R$ to 150 and run the cell. How does the voltage slope change during the stimulus?
---

### 🚫 By-Hand Drill — No AI, No Code
The LIF model has a clean closed-form solution — this is exactly the kind of question a quiz will ask.

Using $R=100$, $\tau=20$ ms, $V_{rest}=-65$ mV, and a constant input current $I=0.5$ nA starting at $t=0$ with $V(0)=V_{rest}$:

By hand, using $V(t) = V_{rest} + RI(1-e^{-t/\tau})$, compute $V$ at $t=\tau$ (i.e., $t=20$ ms), showing every step.

*Your Answer Here (show your work) — does your answer land at ~63.2% of the way from $V_{rest}$ to the steady-state voltage, matching the time-constant definition from Set 1/Set 3?*

In [ ]:
#@title LIF Experimental Rig { run: "auto" }
import numpy as np
import matplotlib.pyplot as plt

# Parameters from sliders
R = 100 #@param {type:"slider", min:50, max:250, step:5}
tau = 0.02 #@param {type:"slider", min:0.005, max:0.1, step:0.001}
V_rest, V_th, V_reset = -65, -50, -70

t = np.arange(0, 1.0, 1e-4)
I = np.zeros_like(t)
I[(t>0.1)&(t<0.6)] = 0.5; I[(t>0.7)&(t<0.8)] = 0.8
V = np.ones_like(t)*V_rest; spikes = []

for k in range(1, len(t)):
    V[k] = V[k-1] + ((-(V[k-1]-V_rest) + R*I[k-1])*(1e-4/tau))
    if V[k] >= V_th:
        spikes.append(t[k]); V[k] = V[k-1] = V_reset

plt.figure(figsize=(8, 3.5))
plt.plot(t, V, 'k'); plt.ylabel('V (mV)')
plt.twinx(); plt.plot(t, I, 'C0', alpha=0.3); plt.ylabel('I (nA)')
plt.title(f'LIF Response | Spikes: {len(spikes)}')
plt.show()

## C2 — Izhikevich Model
### Exercise: Tuning Firing Patterns
This model uses four parameters ($a, b, c, d$) to replicate complex biology.
* **Regular Spiking (RS):** Use the default slider values.
* **Bursting:** Set **c** to -50 and **d** to 2.

---
### Exercises: Adaptation & Bursting
**Exercise 1: Spike Frequency Adaptation**
1. **Predict:** Using the **Regular Spiking (RS)** parameters, will the interval between the 1st and 2nd spike be the same as the interval between the 5th and 6th?
2. **Verify:** Run the simulation. Use the plot to measure the Inter-Spike Interval (ISI) between early spikes versus late spikes.

**Exercise 2: Parameter Sensitivity**
1. **Predict:** The variable $d$ represents the "reset" of the recovery variable $u$. If you decrease $d$ from 8 to 2, will the neuron's firing rate increase or decrease?
2. **Verify:** Change $d=2$ in the RS parameters and run the cell.
3. **Challenge:** Change the parameters to $a=0.02, b=0.2, c=-50, d=2$ (**Bursting**). Identify the time length of the quiet period between bursts.
---

### 🚫 By-Hand Drill — No AI, No Code
The Izhikevich model updates via two coupled equations. Do ONE Euler step by hand — same method as Set 1/Set 3, just two variables instead of one:

$$\dot v = 0.04v^2 + 5v + 140 - u + I \qquad \dot u = a(bv - u)$$

Using Regular Spiking parameters ($a=0.02$, $b=0.2$), with $v=-65$, $u=-13$, $I=10$, and $dt=1$ ms:

By hand, compute $\dot v$ and $\dot u$, then use the Euler update rule ($v_{new} = v + \dot v \cdot dt$) to find $v_{new}$ and $u_{new}$.

*Your Answer Here (show your work):*

In [ ]:
#@title Izhikevich Parameter Tuning { run: "auto" }
import numpy as np
import matplotlib.pyplot as plt

a = 0.02 #@param {type:"slider", min:0.01, max:0.1, step:0.01}
b = 0.2 #@param {type:"slider", min:0.05, max:0.3, step:0.01}
c = -65 #@param {type:"slider", min:-75, max:-45, step:1}
d = 8 #@param {type:"slider", min:0.05, max:10.0, step:0.05}

T = 1000; I = np.zeros(T); I[200:800] = 10
v = -65 * np.ones(T); u = b * v

for k in range(1, T):
    dv = 0.04*v[k-1]**2 + 5*v[k-1] + 140 - u[k-1] + I[k-1]
    du = a*(b*v[k-1] - u[k-1])
    v[k] = v[k-1] + dv; u[k] = u[k-1] + du
    if v[k] >= 30:
        v[k-1]=30; v[k]=c; u[k]+=d

plt.figure(figsize=(8, 3.5)); plt.plot(v, 'k')
plt.title('C2: Interactive Izhikevich'); plt.ylabel('v (mV)'); plt.show()

## C3 — HH via NEURON

### Model Idea

Biophysical $Na^+$ / $K^+$ conductances produce threshold and refractoriness from channel kinetics; waveforms reflect identifiable mechanisms.

### Interactive Exercise: Biophysical Kinetics
Use the sliders in the rig below to explore how ionic conductances dictate spike timing and recovery.

**Exercise 1: Kinetic Latency & Timing**
* **The Goal:** Observe how stimulus strength affects the "time-to-fire."
* **Task:** Move the `stim_delay` to 100ms. Now, slowly increase the `stim_amp` from 0.08 to 0.15 nA.
* **Observe:** Does the first spike move closer to the dotted blue "Onset" line? This represents the time required for sodium channels to reach the open-state threshold.

**Exercise 2: The Refractory Barrier & AHP**
* **The Goal:** Understand Afterhyperpolarization (AHP).
* **Observe:** Look at the "dip" in voltage immediately following a spike.
* **Task:** Adjust `stim_amp` to a high value (e.g., 0.25 nA) to cause rapid firing.
* **Verify:** Note how the cell cannot fire again until the voltage recovers from that dip. This is where $K^+$ gates are still closing, creating the refractory period you measured in the Appendix.

In [ ]:
#@title C3: Hodgkin-Huxley Experimental Rig { run: "auto" }
from neuron import h
import numpy as np
import matplotlib.pyplot as plt

# 1. Clear memory first
h('forall delete_section()')

# 2. Parameters from Sliders
stim_amp = 0.1 #@param {type:"slider", min:0.05, max:0.3, step:0.01}
stim_dur = 600 #@param {type:"slider", min:100, max:800, step:50}
stim_delay = 100 #@param {type:"slider", min:10, max:300, step:10}

# 3. Rig Setup
soma = h.Section(name='soma_hh')
soma.L = soma.diam = 12.6157
soma.insert('hh')

stim = h.IClamp(soma(0.5))
stim.delay = stim_delay
stim.dur = stim_dur
stim.amp = stim_amp

# 4. Recording & Run
v = h.Vector().record(soma(0.5)._ref_v)
t = h.Vector().record(h._ref_t)
h.finitialize(-65)
h.continuerun(800)

# 5. Plotting
plt.figure(figsize=(8, 4))
plt.plot(t, v, 'k', lw=1.5, label='Membrane Voltage')
plt.axhline(-65, color='gray', linestyle='--', alpha=0.5)
plt.axvline(stim_delay, color='C0', linestyle=':', alpha=0.8, label='Stimulus Onset')
plt.title(f'HH Current Clamp | Delay: {stim_delay}ms | Amp: {stim_amp}nA')
plt.xlabel('Time (ms)'); plt.ylabel('Membrane Voltage (mV)')
plt.legend(loc='upper right'); plt.grid(alpha=0.2)
plt.show()

## C4 — Real-world Data Bridge

### Model Idea

In a Brain-Computer Interface (BCI) or sensory prosthetic, we don't use sliders—we use **Sensors**. However, sensors often output "Raw Units" (like 0-1023) that don't match the "Biophysical Units" (nA) of our model.

### Interactive Exercise: The Scaling Challenge
**The Goal:** Map raw sensor data to the `stim.amp` of your HH model to trigger meaningful spikes.

1. **Observe the Input:** The top (gray) plot shows a signal recorded from an external device.
2. **Adjust `data_scale`:** This acts as your "Gain." If it's too low, the signal won't reach the **Rheobase** you discovered in the Appendix.
3. **Adjust `data_offset`:** This moves the entire signal up or down. Use this to set the "Baseline" firing rate.

**Exercise: Precision Triggering**
* **Task:** Adjust the sliders so that the neuron fires **exactly one spike** during the highest peak of the sensor data, but remains silent during the smaller ripples.
* **Reflect:** In a prosthetic limb, why is it critical to tune the `data_offset` so the model doesn't "fire" when the sensor is just reading background noise?

**Make It Your Own (optional AI-assisted extension).** Once precision triggering works, extend the bridge:
- Add a simple **noise filter** (e.g., a moving average) applied to the raw sensor signal before it reaches `data_scale`/`data_offset`, and see whether it changes how easy precision triggering is.
- Add a printed **false-positive counter**: how many spikes fire during the 'quiet' ripples vs. the intended peak?

> **Suggested prompt:** *"Here is my NEURON sensor-to-soma bridge code: [paste it]. I want to add [your extension] without changing how data_scale and data_offset work. Show me what to add and where."*

*Your Answer Here (what you added, and whether it made triggering more or less reliable):*

In [ ]:
#@title C4: Sensor-to-Soma Data Bridge { run: "auto" }
data_scale = 0.005 #@param {type:"slider", min:0.001, max:0.02, step:0.001}
data_offset = 0.05 #@param {type:"slider", min:-0.1, max:0.2, step:0.01}

from neuron import h
import numpy as np
import matplotlib.pyplot as plt

# 1. Create a dummy "Real-world" sensor signal (e.g., an Accelerometer)
t_data = np.linspace(0, 500, 1000)
raw_sensor = 50 + 30 * np.sin(2 * np.pi * 0.01 * t_data) + 10 * np.random.randn(1000)

# 2. Map Sensor -> Model Amps
# We apply the user's scaling and offset here
mapped_amps = (raw_sensor * data_scale) + data_offset

# 3. Setup NEURON Rig
h('forall delete_section()')
soma_c4 = h.Section(name='somac4')
soma_c4.L = soma_c4.diam = 20
soma_c4.insert('hh')

# Use play() to drive the injection with the mapped array
stim_vec = h.Vector(mapped_amps)
t_vec_stim = h.Vector(t_data)
stim = h.IClamp(soma_c4(0.5))
stim.delay = 0
stim.dur = 1e9 # Keep open to follow the vector
stim_vec.play(stim._ref_amp, t_vec_stim, 1)

# Record
v = h.Vector().record(soma_c4(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

h.finitialize(-65)
h.continuerun(500)

# 4. Plotting
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

ax1.plot(t_data, raw_sensor, color='gray', alpha=0.5)
ax1.set_ylabel('Raw Sensor Units')
ax1.set_title('External Data Input')

ax2.plot(t, v, 'k')
ax2.set_ylabel('Membrane V (mV)')
ax2.set_xlabel('Time (ms)')
ax2.axhline(-50, color='red', ls=':', alpha=0.5, label='Threshold')

plt.tight_layout()
plt.show()

### 🎯 Design Problem — Parameter Transparency
**Scenario:** Two models produce identical-looking spike trains for the same input. Model A uses parameters in real biophysical units (mV, nA, ms) — e.g., $g_{Na}=0.12$ S/cm². Model B uses unitless, arbitrary "strength" scores (e.g., "channel strength = 7.3") tuned until its output visually matches Model A's.

**Which model is more scientifically useful, and why?**

A) Model A — biophysical units let you interpret, validate, and compare results against real recordings; Model B's fit could be right for the wrong reasons
B) Model B — fewer constraints means it's more flexible and therefore more useful
C) They're equally useful, since both produce the same output
D) Model A is worse, since real units make the model harder to tune

*Reason it out before revealing the answer.*

**Answer: A).**

Matching output alone doesn't mean matching mechanism. Model B's arbitrary units could be compensating for a completely wrong underlying assumption — there's no way to tell, because there's nothing to check the numbers against. Model A's biophysical units mean every parameter can be independently validated against real measurements (e.g., does $g_{Na}=0.12$ match a real recorded sodium conductance?), and every result can be meaningfully compared to other studies.

This is the exact failure mode described in this Set's Knowledge Bank (Category E): a model can look right and still be right for the wrong reasons.

### 🎯 Your Turn — Design Two Questions, Not Just Answer Them
Pick any concept from this Set. Create **two** short questions of your own — one of each type:

**1. An Analysis-type question:** present a scenario, plot, or result, and ask the reader to interpret *why* something happened or what it implies.

**2. A Design-type question:** ask the reader *how* to achieve a specific target outcome, or what change would produce a given effect.

For each: include a brief setup, 2–4 possible answers, the correct answer, and a short explanation of why. No required format or topic beyond that — design both however makes sense to you. If your questions would genuinely catch someone who only skimmed this Set, you've demonstrated real understanding.

*Your Analysis-type question, answers, and explanation here:*

*Your Design-type question, answers, and explanation here:*

---
## Knowledge Bank — Set 4: Cellular Models & Electrophysiology Basics

**This is the outcome of this module.** Quizzes, the midterm, and the final draw directly from these questions — know them well. For anything you're unsure about, use an AI assistant to research it, and add what you find as your own notes right in this notebook.

### A. Leaky Integrate-and-Fire (LIF)
1. Write the closed-form solution for $V(t)$ under a constant input current, starting from rest.
2. If you double the time constant $\tau$, does the neuron need a higher or lower input frequency to reach threshold? Why?
3. If input resistance $R$ increases, what happens to the voltage slope during a current step, and why?
4. What does the LIF model deliberately leave out, compared to Hodgkin-Huxley?
5. **Geometric scaling:** for a large-diameter neuron (like an alpha motor neuron), how would $R$ and $\tau$ need to change to stay biophysically accurate?

### B. The Izhikevich Model
6. Write the two coupled Izhikevich equations ($\dot v$ and $\dot u$).
7. What do the four parameters $a$, $b$, $c$, $d$ each control?
8. By hand, perform one Euler step given RS parameters, $v=-65$, $u=-13$, $I=10$, $dt=1$ ms.
9. Does the interval between the 1st and 2nd spike equal the interval between the 5th and 6th, under Regular Spiking parameters? What does this reveal about adaptation?
10. How does decreasing $d$ affect firing rate, and why?
11. How does the reset parameter $c$ relate to afterhyperpolarization (AHP)?
12. Biologically, what slow process might the recovery variable $u$ (and specifically $d$) stand in for?

### C. Hodgkin-Huxley via NEURON
13. Why can a Current Clamp not cleanly separate overlapping Na⁺ and K⁺ currents, while a Voltage Clamp can?
14. What does the "dip" immediately following a spike represent, mechanistically?
15. What is the primary advantage of using a biophysical simulator like NEURON over writing your own Euler integration from scratch?
16. Is the I-V curve for a real channel a perfectly straight line? What causes it to curve at higher voltages?

### D. The Real-World Data Bridge (BCI)
17. Why do raw sensor units typically need both a scale (gain) and an offset before driving a biophysical model?
18. Why is tuning the offset so that background noise doesn't trigger false spikes especially important in a prosthetic application?
19. If sensor data is noisy, how might that noise interact with the rheobase threshold?

### E. Model Comparison and Methodology
20. Define a "model abstraction" that preserves interpretability. Which parameters must stay in biophysical units, and why?
21. What is gained and lost moving from a detailed HH-type model to a reduced Izhikevich model?
22. Compare refractory behavior in Izhikevich vs. HH — which felt more "biological" as you varied parameters?
23. Why is "parameter transparency" essential for educational modeling? Give one consequence of opaque/arbitrary-unit scaling.
24. List 3-4 things you should document as assumptions, so a reader can reproduce and critique your model.
25. How would you validate that an interactive rig is working correctly before using it to collect data for a lab report?
26. Give an example of a missing mechanism (e.g., omitting calcium) that could cause a consistent model-data mismatch.
27. Why do we advocate for a "membrane step test" as the first step in characterizing any new neural model?
28. Outline a minimal reproducible workflow: notebook → plots → metrics, from an Appendix measurement to a main-Set simulation.
29. What is the trade-off of adding noise sources to a beginner-level model — does it help or hinder learning core concepts?
30. **Forward Reflection:** You treated Izhikevich's $d$ as a compact stand-in for a slow, calcium-dependent potassium current governing bursting. In Set 5, you'll model that mechanism explicitly through Hodgkin-Huxley gating variables. What do you expect to gain in biological insight, and lose in computational simplicity?